Dataset: https://www.kaggle.com/datasets/asaniczka/1-3m-linkedin-jobs-and-skills-2024

Covers: `02_skills.py` + `03_model.py`

## Libraries

In [ ]:
import os
import re
from collections import Counter
from itertools import combinations

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer

## Paths

In [ ]:
KAGGLE_DIR = "/kaggle/input/1-3m-linkedin-jobs-and-skills-2024"
LOCAL_DIR = "."

data_dir = KAGGLE_DIR if os.path.exists(KAGGLE_DIR) else LOCAL_DIR

postings_path = f"{data_dir}/linkedin_job_postings.csv"
skills_path = f"{data_dir}/job_skills.csv"
summary_path = f"{data_dir}/job_summary.csv"

PARQUET_PATH = "/tmp/data/merged.parquet"
SAMPLE_N = 200_000

print("data_dir:", data_dir)

## Shared Contracts — processor.py

In [ ]:
def load_postings(path: str, sample_n: int) -> pd.DataFrame:
    df = pd.read_csv(path, nrows=sample_n)
    df["job_title"] = df["job_title"].astype(str).str.lower().str.strip()
    df["company"] = df["company"].fillna("Unknown Company")
    df["job_location"] = df["job_location"].fillna("Unknown Location")
    df = df.drop(columns=["got_summary", "got_ner", "is_being_worked"], errors="ignore")
    df = df.drop_duplicates(subset=[c for c in df.columns if c != "job_link"])
    processed = pd.to_datetime(
        df["last_processed_time"], dayfirst=True, format="mixed", utc=True
    )
    df["date"] = processed.dt.date
    df["hour"] = processed.dt.hour
    df["day"] = processed.dt.day
    df["day_of_week"] = processed.dt.day_name()
    df = df.drop(columns=["last_processed_time"])
    return df


def aggregate_skills(path: str) -> pd.DataFrame:
    skills_raw = pd.read_csv(path)
    return (
        skills_raw.groupby("job_link")["job_skills"]
        .apply(list)
        .reset_index()
        .rename(columns={"job_skills": "skills_list"})
    )


def load_summary(path: str) -> pd.DataFrame:
    return pd.read_csv(path, usecols=["job_link", "job_summary"])


def merge_datasets(
    postings: pd.DataFrame,
    skills_agg: pd.DataFrame,
    summary: pd.DataFrame,
) -> pd.DataFrame:
    df = postings.merge(skills_agg, on="job_link", how="left")
    df = df.merge(summary, on="job_link", how="left")
    df["skills_list"] = df["skills_list"].apply(
        lambda x: x if isinstance(x, list) else []
    )
    df["job_summary"] = df["job_summary"].fillna("")
    return df


CATEGORY_KEYWORDS: dict[str, list[str]] = {
    "TECHNOLOGY": [
        "software engineer",
        "software developer",
        "backend",
        "frontend",
        "full stack",
        "fullstack",
        "devops",
        "cloud engineer",
        "platform engineer",
        "site reliability",
        "mobile developer",
        "ios developer",
        "android developer",
        "systems engineer",
        "network engineer",
        "security engineer",
        "cybersecurity",
        "infrastructure engineer",
        "embedded",
        "firmware",
        "it engineer",
        "it specialist",
        "programmer",
    ],
    "DATA-ANALYTICS": [
        "data analyst",
        "data scientist",
        "data engineer",
        "analytics engineer",
        "business analyst",
        "business intelligence",
        "bi developer",
        "reporting analyst",
        "quantitative analyst",
        "research analyst",
        "data architect",
        "machine learning engineer",
        "ml engineer",
        "ai engineer",
        "nlp engineer",
        "data warehouse",
    ],
    "MARKETING": [
        "marketing manager",
        "marketing specialist",
        "marketing coordinator",
        "seo",
        "sem",
        "content strategist",
        "content writer",
        "brand manager",
        "digital marketing",
        "social media manager",
        "growth manager",
        "campaign manager",
        "communications manager",
        "public relations",
        "copywriter",
        "creative director",
        "email marketing",
    ],
    "SALES": [
        "sales representative",
        "sales rep",
        "account executive",
        "business development representative",
        "bdr",
        "sdr",
        "sales development",
        "sales engineer",
        "solutions engineer",
        "inside sales",
        "outside sales",
        "territory manager",
        "regional sales manager",
        "sales manager",
        "sales director",
    ],
    "FINANCE": [
        "financial analyst",
        "finance analyst",
        "accountant",
        "accounting manager",
        "controller",
        "fp&a",
        "financial planning",
        "auditor",
        "tax analyst",
        "treasury analyst",
        "investment analyst",
        "portfolio manager",
        "risk analyst",
        "credit analyst",
        "actuary",
        "finance manager",
        "chief financial",
    ],
    "HR-OPERATIONS": [
        "human resources",
        "hr manager",
        "hr generalist",
        "hr business partner",
        "hr coordinator",
        "recruiter",
        "talent acquisition",
        "people operations",
        "operations manager",
        "operations analyst",
        "supply chain",
        "logistics manager",
        "procurement",
        "warehouse manager",
        "fulfillment manager",
        "compensation",
        "payroll",
    ],
    "PRODUCT-DESIGN": [
        "product manager",
        "product owner",
        "ux designer",
        "ui designer",
        "ux researcher",
        "product designer",
        "user experience",
        "user interface",
        "interaction designer",
        "visual designer",
        "graphic designer",
        "product lead",
        "head of product",
    ],
    "CUSTOMER-SUCCESS": [
        "customer success",
        "customer support",
        "customer service manager",
        "client success",
        "client services",
        "support engineer",
        "technical support",
        "help desk",
        "customer experience",
        "service desk",
        "cx manager",
    ],
}


def derive_category(title: str) -> str | None:
    t = title.lower()
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in t for kw in keywords):
            return category
    return None


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in ["job_title", "job_summary"]:
        out[f"{col}_len"] = out[col].astype(str).str.len()
        out[f"{col}_word_count"] = out[col].astype(str).str.split().str.len()

    out["n_skills"] = out["skills_list"].apply(len)

    skills_text = out["skills_list"].apply(
        lambda xs: " ".join(str(x).strip() for x in xs if str(x).strip())
    )
    out["combined_text"] = (
        out["job_title"].fillna("")
        + "\n"
        + out["job_summary"].fillna("")
        + "\nskills: "
        + skills_text
    )

    out["skills_norm"] = out["skills_list"].apply(
        lambda xs: [
            re.sub(r"\s+", " ", str(x).strip()).lower() for x in xs if str(x).strip()
        ]
    )

    out["category"] = out["job_title"].apply(derive_category)

    return out


def get_merged(parquet_path: str, sample_n: int) -> pd.DataFrame:
    if os.path.exists(parquet_path):
        return pd.read_parquet(parquet_path)
    postings = load_postings(postings_path, sample_n)
    skills_agg = aggregate_skills(skills_path)
    summary = load_summary(summary_path)
    df = merge_datasets(postings, skills_agg, summary)
    os.makedirs(os.path.dirname(parquet_path), exist_ok=True)
    df.to_parquet(parquet_path, index=False)
    return df

## Load & Build Features

In [ ]:
df = get_merged(PARQUET_PATH, SAMPLE_N)
jobs_eda = build_features(df)

print("shape:", jobs_eda.shape)

## Coverage Check

Review unmatched titles and expand `CATEGORY_KEYWORDS` if coverage is too low.

In [ ]:
total = len(jobs_eda)
matched = jobs_eda["category"].notna().sum()

print(f"Total rows:  {total:,}")
print(f"Matched:     {matched:,}  ({matched / total:.1%})")
print()
print("Category distribution:")
print(jobs_eda["category"].value_counts())
print()
print("Top 30 unmatched job titles (expand keyword map if needed):")
print(jobs_eda[jobs_eda["category"].isna()]["job_title"].value_counts().head(30))

## Filter to Matched Rows

In [ ]:
jobs_eda = jobs_eda[jobs_eda["category"].notna()].copy().reset_index(drop=True)
print("After filter:", jobs_eda.shape)

## Visualization Theme

In [ ]:
bright_palette = [
    "#4CC9F0",
    "#4895EF",
    "#4361EE",
    "#7B2CBF",
    "#9D4EDD",
    "#C77DFF",
    "#F72585",
    "#480CA8",
]

sns.set_theme(style="whitegrid", context="talk", palette=bright_palette)
plt.rcParams["figure.facecolor"] = "#F8FAFF"
plt.rcParams["axes.facecolor"] = "#FFFFFF"
plt.rcParams["axes.edgecolor"] = "#D6D6E5"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["grid.linestyle"] = "--"

## 02_skills: EDA Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

order = jobs_eda["category"].value_counts().index
sns.countplot(
    data=jobs_eda, x="category", order=order, ax=axes[0], palette=bright_palette
)
axes[0].set_title("Category counts")
axes[0].tick_params(axis="x", rotation=45)

sns.histplot(
    jobs_eda["job_summary_word_count"], bins=40, kde=True, ax=axes[1], color="#4CC9F0"
)
axes[1].set_title("Job summary word count distribution")

sns.histplot(
    jobs_eda["job_title_word_count"], bins=25, kde=True, ax=axes[2], color="#4895EF"
)
axes[2].set_title("Job title word count distribution")

sns.histplot(jobs_eda["n_skills"], bins=30, kde=False, ax=axes[3], color="#9D4EDD")
axes[3].set_title("# Skills per job distribution")

plt.tight_layout()
plt.show()

## 02_skills: Top N Skills by Frequency

In [ ]:
all_skills = [s for xs in jobs_eda["skills_norm"] for s in xs]
skill_counts = Counter(all_skills)
top_skills_overall = pd.DataFrame(
    skill_counts.most_common(30), columns=["skill", "count"]
)

plt.figure(figsize=(10, 8))
sns.barplot(data=top_skills_overall.head(25), x="count", y="skill", color="#9D4EDD")
plt.title("Top 25 Skills (overall)")
plt.xlabel("Count")
plt.ylabel("Skill")
plt.tight_layout()
plt.show()

## 02_skills: Skills Breakdown by Category

In [ ]:
rows = []
for cat, grp in jobs_eda.groupby("category"):
    c = Counter(s for xs in grp["skills_norm"] for s in xs)
    for skill, cnt in c.most_common(12):
        rows.append({"category": cat, "skill": skill, "count": cnt})

top_by_category = pd.DataFrame(rows).sort_values(
    ["category", "count"], ascending=[True, True]
)

g = sns.catplot(
    data=top_by_category,
    x="count",
    y="skill",
    col="category",
    col_wrap=3,
    kind="bar",
    height=7,
    aspect=1.2,
    sharex=False,
    sharey=False,
    color="#9D4EDD",
    edgecolor="white",
    linewidth=0.7,
)
g.set_titles("{col_name}", fontweight="bold")
g.set_axis_labels("Count", "Skill")
g.figure.subplots_adjust(wspace=0.28, hspace=0.35)
plt.show()

## 02_skills: Skill Co-occurrence Pairs

In [ ]:
pair_counts: Counter = Counter()
for xs in jobs_eda["skills_norm"]:
    uniq = sorted(set(x for x in xs if x))[:40]
    for a, b in combinations(uniq, 2):
        pair_counts[(a, b)] += 1

top_pairs = pd.DataFrame(
    [
        {"skill_a": a, "skill_b": b, "count": c}
        for (a, b), c in pair_counts.most_common(30)
    ]
)

pair_labels = top_pairs.head(15).apply(
    lambda r: f"{r['skill_a']} + {r['skill_b']}", axis=1
)

plt.figure(figsize=(10, 8))
sns.barplot(data=top_pairs.head(15), x="count", y=pair_labels, color="#9D4EDD")
plt.title("Top 15 Co-occurring Skill Pairs")
plt.xlabel("Co-occurrence count")
plt.ylabel("Skill pair")
plt.tight_layout()
plt.show()

top_pairs.head(15)

## 03_model: Baseline (TF-IDF + Logistic Regression)

In [ ]:
assert "category" in jobs_eda.columns, "Missing target column: category"

X = jobs_eda["combined_text"].astype(str)
y = jobs_eda["category"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

baseline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                stop_words="english",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.9,
                max_features=50000,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                solver="saga",
            ),
        ),
    ]
)

baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)

print("Baseline classification report:\n")
print(classification_report(y_test, pred))

## 03_model: GridSearchCV

In [ ]:
param_grid = {
    "tfidf__max_features": [20000, 50000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__C": [0.5, 1.0, 2.0],
}

search = GridSearchCV(
    estimator=baseline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV f1_macro:", search.best_score_)

best_model = search.best_estimator_
final_pred = best_model.predict(X_test)

print("\nFinal (test) classification report:\n")
print(classification_report(y_test, final_pred))

## 03_model: Confusion Matrix & Error Analysis

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, final_pred, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"true:{l}" for l in labels],
    columns=[f"pred:{l}" for l in labels],
)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="BuPu", linewidths=0.5, linecolor="white")
plt.title("Confusion Matrix (Best Model)")
plt.tight_layout()
plt.show()

errors = pd.DataFrame(
    {
        "text": X_test.values,
        "y_true": y_test.values,
        "y_pred": final_pred,
    }
)
errors = errors[errors["y_true"] != errors["y_pred"]].copy()
print(f"# errors: {len(errors)}")
errors.sample(min(10, len(errors)), random_state=42)

## 03_model: Save Model

Output: `assets/models/baseline.pkl`

In [ ]:
joblib.dump(best_model, "baseline.pkl")
print("Saved baseline.pkl")

## Skills: KMeans Clustering

In [ ]:
mlb = MultiLabelBinarizer(sparse_output=True)
X_skills = mlb.fit_transform(jobs_eda["skills_norm"])
skill_names = np.array(mlb.classes_)

k = 8
kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
clusters = kmeans.fit_predict(X_skills)

jobs_eda = jobs_eda.copy()
jobs_eda["skill_cluster"] = clusters

centers = kmeans.cluster_centers_
cluster_summaries = []
for i in range(k):
    top_idx = np.argsort(centers[i])[::-1][:15]
    cluster_summaries.append(
        {
            "cluster": i,
            "size": int((clusters == i).sum()),
            "top_skills": ", ".join(skill_names[top_idx]),
        }
    )

cluster_summary_df = pd.DataFrame(cluster_summaries).sort_values(
    "size", ascending=False
)
print(cluster_summary_df.to_string())

ct = pd.crosstab(jobs_eda["skill_cluster"], jobs_eda["category"], normalize="index")
print("\nCluster-category breakdown:")
print(ct.round(2).to_string())

plt.figure(figsize=(10, 4))
sns.countplot(
    data=jobs_eda,
    x="skill_cluster",
    order=sorted(jobs_eda["skill_cluster"].unique()),
    color="#9D4EDD",
)
plt.title("Cluster sizes (skills-based)")
plt.tight_layout()
plt.show()

## Course Recommendations

In [ ]:
def propose_courses(
    top_by_category_df: pd.DataFrame,
    top_pairs_df: pd.DataFrame,
    cluster_df: pd.DataFrame,
    top_skills_df: pd.DataFrame,
) -> pd.DataFrame:
    courses = []

    overall_core = top_skills_df.head(12)["skill"].tolist()
    courses.append(
        {
            "course_track": "Core professional skills (cross-domain)",
            "why": "High-frequency cross-cutting skills across all categories and clusters",
            "skills_covered": ", ".join(overall_core),
        }
    )

    for cat in sorted(top_by_category_df["category"].unique()):
        sk = (
            top_by_category_df[top_by_category_df["category"] == cat]
            .sort_values("count", ascending=False)
            .head(12)["skill"]
            .tolist()
        )
        courses.append(
            {
                "course_track": f"{cat} track (job-ready)",
                "why": "Most frequently required skills for this job category",
                "skills_covered": ", ".join(sk),
            }
        )

    pairs = (
        top_pairs_df.head(10)
        .apply(lambda r: f"{r['skill_a']} + {r['skill_b']}", axis=1)
        .tolist()
    )
    courses.append(
        {
            "course_track": "Stackable micro-credentials (skill bundles)",
            "why": "Frequently co-occurring skill pairs — ideal for stackable micro-credentials",
            "skills_covered": " | ".join(pairs),
        }
    )

    for _, row in cluster_df.sort_values("size", ascending=False).head(4).iterrows():
        courses.append(
            {
                "course_track": f"Persona bundle (cluster {int(row['cluster'])})",
                "why": "Typical job competency profile extracted from skill co-occurrence structure",
                "skills_covered": row["top_skills"],
            }
        )

    return pd.DataFrame(courses)


courses_df = propose_courses(
    top_by_category, top_pairs, cluster_summary_df, top_skills_overall
)
courses_df.to_csv("course_recommendations_from_job_posts.csv", index=False)
print("Saved: course_recommendations_from_job_posts.csv")
courses_df